In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42  
TRAIN_PATH = "../../../src/data/classificationdata/H2_classification_train_with_agent.csv"
TEST_PATH = "../../../src/data/classificationdata/H2_classification_test_with_agent.csv"
TARGET = "IsCanceled"

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

print("train:", train.shape)
print("test :", test.shape)

train: (43332, 55)
test : (12343, 55)


In [3]:
# #Cell 3: basic checks
# The first line checks both files have the same columns in the same order. If not, the model will break.
# isna().sum().sum() counts empty cells. We want 0.
# duplicated().sum() counts repeated rows.
# value_counts(normalize=True) shows the share of 0s and 1s. About 42% of the original data is canceled. If the rates differ a lot between files, something is wrong.
print("Same columns in both files:", list(train.columns) == list(test.columns))
print("Missing values  train:", train.isna().sum().sum(), " test:", test.isna().sum().sum())
print("Duplicate rows  train:", train.duplicated().sum(), " test:", test.duplicated().sum())

print("\nCancel rate (1 = canceled)")
print("train:", train[TARGET].value_counts(normalize=True).round(3).to_dict())
print("test :", test[TARGET].value_counts(normalize=True).round(3).to_dict())

Same columns in both files: True
Missing values  train: 0  test: 0
Duplicate rows  train: 653  test: 367

Cancel rate (1 = canceled)
train: {0: 0.698, 1: 0.302}
test : {0: 0.675, 1: 0.325}


In [4]:
overlap = train.merge(test, how="inner")   # rows that appear in both files
print("Rows that appear in both train and test:", len(overlap))

Rows that appear in both train and test: 89


In [5]:
X_train = train.drop(columns=[TARGET, "Agent_te"])   # Agent_te dropped on purpose
y_train = train[TARGET]

X_test  = test.drop(columns=[TARGET, "Agent_te"])
y_test  = test[TARGET]

X_train["Agent"] = X_train["Agent"].astype(str)      # agent IDs must be text
X_test["Agent"]  = X_test["Agent"].astype(str)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
print(list(X_train.columns))

(43332, 53) (43332,)
(12343, 53) (12343,)
['LeadTime', 'TotalOfSpecialRequests', 'month_1', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12', 'TotalNights', 'WeekendOrNo', 'IsZeroNights', 'total_guests', 'Meal_BB', 'Meal_FB', 'Meal_HB', 'Meal_SC', 'Country_AUT', 'Country_BEL', 'Country_BRA', 'Country_CHE', 'Country_CHN', 'Country_DEU', 'Country_ESP', 'Country_FRA', 'Country_GBR', 'Country_IRL', 'Country_ITA', 'Country_NLD', 'Country_PRT', 'Country_USA', 'Country_infrequent_sklearn', 'MarketSegment_Aviation', 'MarketSegment_Complementary', 'MarketSegment_Corporate', 'MarketSegment_Direct', 'MarketSegment_Groups', 'MarketSegment_Offline TA/TO', 'MarketSegment_Online TA', 'has_prev_not_canceled', 'DepositType_No Deposit', 'DepositType_Non Refund', 'DepositType_Refundable', 'has_company', 'CustomerType_Transient_Party', 'CustomerType_Contract', 'CustomerType_Group', 'Agent']


In [6]:
#Which columns are NOT 0/1? These show whether the numbers are scaled already
#Models like KNN, SVM and Logistic Regression need scaled numbers. 
#If LeadTime goes up to hundreds, we scale it inside the pipeline. If it is already small, the team scaled it
not_binary = [c for c in X_train.columns if X_train[c].nunique() > 2]
print(X_train[not_binary].describe().round(2).T[["min", "mean", "max"]])

                        min   mean    max
LeadTime                0.0  78.49  629.0
TotalOfSpecialRequests  0.0   0.71    5.0
TotalNights             0.0   3.14   48.0
total_guests            1.0   2.02   12.0


In [7]:
for col in ["DepositType_No Deposit", "DepositType_Non Refund", "DepositType_Refundable"]:
    rate = y_train[X_train[col] == 1].mean()
    count = int(X_train[col].sum())
    print(f"{col}: {count} bookings, cancel rate {rate:.2f}")

DepositType_No Deposit: 42532 bookings, cancel rate 0.29
DepositType_Non Refund: 787 bookings, cancel rate 0.98
DepositType_Refundable: 13 bookings, cancel rate 0.69


In [8]:
# Compare all models with 5-fold cross-validation on the train file only. 
# The train data is cut into 5 parts, and each part takes a turn as the "exam" while the model learns from the other 4.
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold

cv  = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED)  # 10 fits # This gives 10 scores per model instead of 5. Runs take twice as long
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)         # 5 fits

In [9]:
# a helper function that scores any model
import time
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.metrics import make_scorer, recall_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, StandardScaler

def make_pipe(model, scale=False):
    enc = ColumnTransformer(
        [("agent", TargetEncoder(target_type="binary", smooth="auto",
                                 cv=5, random_state=SEED), ["Agent"])],
        remainder="passthrough")                 # all other columns pass through
    steps = [("enc", enc)]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

def weighted_hm(y_true, y_pred, w_recall=2, w_spec=1):
    recall = recall_score(y_true, y_pred, pos_label=1)
    spec = recall_score(y_true, y_pred, pos_label=0) #Specificity=TN/TN+FP
    if recall == 0 or spec == 0:
        return 0.0
    return (w_recall + w_spec) / (w_recall / recall + w_spec / spec)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "hm_1_1": make_scorer(weighted_hm, w_recall=1, w_spec=1),#Recall and Specificity have equal weights
    "hm_2_1": make_scorer(weighted_hm, w_recall=2, w_spec=1),#Recall gets twice the weight
    "hm_3_1": make_scorer(weighted_hm, w_recall=3, w_spec=1)#Recall gets three times the weight
}

def run_cv(name, model, scale=False, cv_obj=None):
    """Cross-validate one model on the train data and return a result row."""
    cv_obj = cv_obj if cv_obj is not None else cv   # default: the repeated CV
    
    model = make_pipe(model, scale)

    start = time.time()
    res = cross_validate(model, X_train, y_train, cv=cv_obj, scoring=scoring, n_jobs=-1)
    total_time = time.time() - start

    row = {"model": name}
    for metric in scoring:
        row[f"cv_{metric}"] = res[f"test_{metric}"].mean()
    row["cv_f1_std"] = res["test_f1"].std()
    row["time_sec"] = total_time
    row["cv_scheme"] = f"{len(res['test_f1'])} fits"   # 10 = repeated, 5 = single
    return row

In [10]:
#test helper function for one model
from sklearn.linear_model import LogisticRegression

row = run_cv("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED), scale=True)
pd.DataFrame([row]).round(4)

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_hm_1_1,cv_hm_2_1,cv_hm_3_1,cv_f1_std,time_sec,cv_scheme
0,LogisticRegression,0.7678,0.6644,0.4683,0.5493,0.811,0.6575,0.6154,0.5571,0.5319,0.0093,6.0983,10 fits


In [11]:
#Train all 8 candidate models with default settings and compare them by cross-validation
#Linear Regression (baseline). 
# K-Nearest Neighbors Regressor. 
# Decision Tree Regressor. 
# Random Forest Regressor. 
# XGBoost Regressor or another approved gradient-boosted tree implementation. 
# Multilayer Perceptron / Neural Network Regressor.
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# name -> (model, needs scaling?)
models = {
    "GaussianNB":         (GaussianNB(), False),
    "LogisticRegression": (LogisticRegression(max_iter=1000, random_state=SEED), True),
    "KNN":                (KNeighborsClassifier(), True),
    "DecisionTree":       (DecisionTreeClassifier(random_state=SEED), False),
    "RandomForest":       (RandomForestClassifier(random_state=SEED, n_jobs=-1), False),
    "XGBoost":            (XGBClassifier(random_state=SEED, eval_metric="logloss"), False),
    "MLP":                (MLPClassifier(max_iter=300, random_state=SEED), True),
}

rows = []
for name, (model, scale) in models.items():
    print("running", name, "...", flush=True)
    # if name == "MLP":
    #     rows.append(
    #         run_cv(name, model, scale=scale, cv_obj=cv5)
    #     )
    # else:
    rows.append(
            run_cv(name, model, scale=scale)
        )

running GaussianNB ...
running LogisticRegression ...
running KNN ...
running DecisionTree ...
running RandomForest ...
running XGBoost ...
running MLP ...


In [12]:
#SVM
from sklearn.svm import SVC

# SVM is slow on 43,000 rows, so it gets its own cell.
rows.append(run_cv("SVM_rbf", SVC(kernel="rbf", random_state=SEED), scale=True))

In [13]:
from pathlib import Path

baseline = pd.DataFrame(rows).sort_values("cv_pr_auc", ascending=False).round(4)
display(baseline)

Path("../../../outputs").mkdir(parents=True, exist_ok=True)
baseline.to_csv("../../../outputs/classification_Cross-validation_Results_AfterAgent.csv", index=False)

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,cv_hm_1_1,cv_hm_2_1,cv_hm_3_1,cv_f1_std,time_sec,cv_scheme
5,XGBoost,0.8049,0.7067,0.6059,0.6524,0.8678,0.7432,0.7213,0.6782,0.6586,0.0073,4.0372,10 fits
4,RandomForest,0.8019,0.7055,0.5914,0.6434,0.8602,0.7281,0.7115,0.6664,0.6459,0.0073,13.1867,10 fits
6,MLP,0.7854,0.6611,0.5959,0.6264,0.8432,0.7025,0.7060,0.6650,0.6462,0.0128,162.5967,10 fits
7,SVM_rbf,0.7903,0.7094,0.5186,0.5991,0.8363,0.6994,0.6601,0.6051,0.5809,0.0076,382.7568,10 fits
1,LogisticRegression,0.7678,0.6644,0.4683,0.5493,0.8110,0.6575,0.6154,0.5571,0.5319,0.0093,1.0585,10 fits
2,KNN,0.7521,0.6007,0.5360,0.5664,0.7793,0.5710,0.6560,0.6105,0.5900,0.0108,11.5282,10 fits
0,GaussianNB,0.6732,0.4718,0.6518,0.5464,0.7228,0.5476,0.6649,0.6601,0.6578,0.0065,0.9550,10 fits
3,DecisionTree,0.7496,0.5854,0.5881,0.5867,0.7044,0.4697,0.6846,0.6491,0.6327,0.0108,1.1152,10 fits
